FAKE NEWS DETECTOR

In [2]:
!pip install pandas numpy sentence-transformers lightgbm scikit-learn xgboost tavily-python google-generativeai fastapi uvicorn python-dotenv

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 4.7 MB/s  0:00:00


In [ ]:
import os
import re
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------------------------------------------
# 1. LOAD ALL RAW CSV FILES
# -------------------------------------------------------------------
raw_dir = '../raw_data'
csv_files = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]
dfs = []

print(f"Reading CSV files: {csv_files}")

for file in csv_files:
    file_path = os.path.join(raw_dir, file)
    df = pd.read_csv(file_path)
    
    # Standardize label column name
    if 'class' in df.columns and 'label' not in df.columns:
        df = df.rename(columns={'class': 'label'})
    
    dfs.append(df)

# Combine into one DataFrame
df_all = pd.concat(dfs, ignore_index=True).dropna(subset=['label'])
df_all['label'] = df_all['label'].astype(int)

# -------------------------------------------------------------------
# 2. COMBINE TITLE & TEXT INTO ONE CLEAN COLUMN
# -------------------------------------------------------------------
title_col = df_all['title'].fillna('') if 'title' in df_all.columns else ""
text_col = df_all['text'].fillna('') if 'text' in df_all.columns else ""

# Merge title + text
df_all['full_text'] = (title_col + " " + text_col).str.strip()

# Clean wire-service signatures (like Reuters / AP)
def clean_text(t):
    t = re.sub(r'^[A-Z\s,]+?\s*\((?:Reuters|AP|AFP|Bloomberg)\)\s*[-—–]\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^[A-Z\s,]+?\s*[-—–]\s*', '', t, flags=re.IGNORECASE)
    return t.strip()

df_all['clean_text'] = df_all['full_text'].apply(clean_text)

# Drop any empty text rows
df_all = df_all[df_all['clean_text'].str.len() > 10].reset_index(drop=True)

# -------------------------------------------------------------------
# 3. SPLIT INTO TRAIN & TEST
# -------------------------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_all['clean_text'].tolist(), 
    df_all['label'].values, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_all['label'].values
)

print(f"Train samples: {len(X_train_text)} | Test samples: {len(X_test_text)}")

# -------------------------------------------------------------------
# 4. GET EMBEDDINGS (SentenceTransformer)
# -------------------------------------------------------------------
print("\nLoading SentenceTransformer ('all-MiniLM-L6-v2')...")
encoder = SentenceTransformer('all-MiniLM-L6-v2')

print("Converting Train text to vector embeddings...")
X_train_emb = encoder.encode(X_train_text, batch_size=128, show_progress_bar=True, normalize_embeddings=True)

print("Converting Test text to vector embeddings...")
X_test_emb = encoder.encode(X_test_text, batch_size=128, show_progress_bar=True, normalize_embeddings=True)

# -------------------------------------------------------------------
# 5. TRAIN XGBOOST MODEL
# -------------------------------------------------------------------
print("\nTraining XGBoost Classifier...")
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_emb, y_train)

# -------------------------------------------------------------------
# 6. EVALUATE & SAVE
# -------------------------------------------------------------------
y_pred = model.predict(X_test_emb)
acc = accuracy_score(y_test, y_pred)

print(f"\n✅ SUCCESS! Model Accuracy: {acc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Fake (0)', 'Real (1)']))

# Save trained XGBoost model
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/xgboost_fake_news.pkl')
print("Model saved to ../models/xgboost_fake_news.pkl")

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading CSV files: ['dataset2_welfake_clean.csv', 'news.csv', 'news_dataset.csv']
Train samples: 62656 | Test samples: 15665

Loading SentenceTransformer ('all-MiniLM-L6-v2')...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3361.47it/s]


Converting Train text to vector embeddings...


Batches: 100%|██████████| 490/490 [39:30<00:00,  4.84s/it]


Converting Test text to vector embeddings...


Batches: 100%|██████████| 123/123 [09:21<00:00,  4.57s/it]



Training XGBoost Classifier...

✅ SUCCESS! Model Accuracy: 0.9033

              precision    recall  f1-score   support

    Fake (0)       0.91      0.90      0.90      7768
    Real (1)       0.90      0.91      0.90      7897

    accuracy                           0.90     15665
   macro avg       0.90      0.90      0.90     15665
weighted avg       0.90      0.90      0.90     15665

Model saved to ../models/xgboost_fake_news.pkl
